<h3> Parsing </h3>

In [1]:
from unstructured.partition.text import partition_text #https://docs.unstructured.io/open-source/core-functionality/partitioning
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.docx import partition_docx

from unstructured.chunking.title import chunk_by_title #https://docs.unstructured.io/open-source/core-functionality/chunking
from unstructured.chunking.basic import chunk_elements

from pathlib import Path
from langchain_core.documents import Document

c:\Users\Admin\Desktop\ip\How To RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs")

documentos = []
chunk_id = 1

for docs in path.iterdir():

    file_dtype = docs.suffix.lower()

    if file_dtype == ".pdf":

        parsing = partition_pdf (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)

    elif file_dtype == ".txt":

        parsing = partition_text (str(docs), languages = ["por"])
        chunks = chunk_elements (parsing, max_characters = 750)

    elif file_dtype == ".docx":        
        
        parsing = partition_docx (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)


    for chunk in chunks:

        documentos.append (
            Document (
                page_content = chunk.text,
                metadata = {
                    "title": docs.name,
                    "chunk_id": chunk_id
                }
            )
        )

        chunk_id += 1


In [ ]:
#documentos

<h3> Embeddings </h3>

In [3]:
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface

C:\Users\Admin\AppData\Local\Temp\ipykernel_12080\2595397039.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores


In [4]:
emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Bi Encoder")

db = FAISS.from_documents (documentos, emb_hf)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1491.99it/s]


In [5]:
import json

with open (r"C:\Users\Admin\Desktop\ip\How To RAG\eval\dataset2.json", "r", encoding = "utf-8") as file:
    dados = json.load (file)

<h3> Retrieval</h3>

<h5> Sparse Retrieval </h5>

In [6]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

sparse_retrieval = BM25Retriever.from_documents (documentos, k = 10)

<h6> eval </h6>

In [7]:
from eval.recallk import recall_k_sparse_retrieval

x = recall_k_sparse_retrieval (sparse_retrieval, dados)

Recall@K Chunk -> 0.6015625
Recall@K Docs -> 0.9921875


<h5> Dense Retrieval </h5>

In [8]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 20
    }
)

#x = dense_retrieval.invoke ("O que é a corrente dinâmica estipulada (Idyn)?")

In [9]:
from eval.recallk import recall_k_dense_retrieval

y = recall_k_dense_retrieval (dense_retrieval, dados)

Recall@K Chunk -> 0.58984375
Recall@K Docs -> 0.9296875


<h5> Reciprocal Rank Fusion </h5>

In [ ]:
#def Reciprocal_Rank_Fusion (rankings, k = 50): #60 é uma variável usada para reduzir o impacto de diferenças pequenas entre ranks
    
    points = {}
    title_doc = {}
    doc_id = {}

    for ranking in rankings:
        #print (ranking)
        for rank, doc in enumerate (ranking):
            
            doc_content = doc.page_content
            doc_title = doc.metadata["title"]
            id_doc = doc.metadata["chunk_id"]

            points[doc_content] = points.get(doc_content, 0) + 1 / (k + rank + 1)

            title_doc[doc_content] = doc_title
            doc_id[doc_content] = id_doc 

    ranking_final = sorted(points.items(), key = lambda x : x[1], reverse = True)


            #points[doc_content] = points.get(doc_content, 0) + 1 / (k + rank + 1)
    #Mudanças para poder englobar o nome dos docs. Importante para calcular Recall@K
    return [
        (title_doc[content], doc_id[content], content, score,)
        for content, score in ranking_final
    ]
    # return sorted (points.items(), key = lambda x: x[1], reverse = False) #Dá return dos docs ordenados e respetivas pontuações sorted

In [12]:
def Reciprocal_Rank_Fusion (rankings, k = 60):
    
    scores = {}
    metadata = {}

    for ranking in rankings:
        for rank, doc in enumerate(ranking):

            doc_id = doc.metadata["chunk_id"]

            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)

            metadata[doc_id] = {
                "title": doc.metadata.get("title"),
                "content": doc.page_content
            }

    ranked = sorted(scores.items(), key = lambda x: x[1], reverse = True)

    return [(
            metadata[doc_id]["title"], 
            doc_id, 
            metadata[doc_id]["content"], 
            score
            )
        for doc_id, score in ranked
    ]

In [13]:
### EXEMPLO

query = "Em que consiste o relatório da rede EU Kids Online ?"

s_retrieval = sparse_retrieval.invoke (query)
d_retrieval = dense_retrieval.invoke (query)

teste = Reciprocal_Rank_Fusion ([s_retrieval, d_retrieval])

display (teste)

[('Crianças_e_jovens_(9-17)_e_Inteligência_Artificial_Generativa_em_Portugal.pdf',
  222,
  'A Rede EU KIDS ONLINE e o estudo por detrás deste relatório\n\nA rede EU Kids Online, uma rede colaborativa e independente, reúne investigadores de 26 países. Sob uma perspetiva multidisciplinar e multimetodológica, investigam oportunidades, riscos e segurança de crianças e jovens na internet. Ver www.eukidsonline.net',
  0.032018442622950824),
 ('Crianças_e_jovens_(9-17)_e_Inteligência_Artificial_Generativa_em_Portugal.pdf',
  204,
  'Palavras-chave: IA Generativa; Crianças e Jovens; EU Kids Online Keywords: AI Gen; Children and youth; EU Kids Online\n\nIdioma | Language: Português | Portuguese Número de páginas | Number of pages: 57 DOI: xx Ano | Year : 2026\n\nComo citar | How to cite\n\nPonte, C., Batista, S., & Luna, E. (2026). Crianças e jovens (9-17 anos) e Inteligência Artificial Generativa em Portugal. Resultados nacionais dos estudos EU Kids Online, 2025. EU Kids Online. Doi:_________

In [10]:
from eval.recallk import recall_k_hybrid_retrieval

z = recall_k_hybrid_retrieval (sparse_retrieval, dense_retrieval, dados)

Recall@K Chunk -> 0.7890625
Recall@K Docs -> 1.0


<h3> Reranking </h3>

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder = AutoModelForSequenceClassification.from_pretrained (path)
ce_tokenization = AutoTokenizer.from_pretrained (path)

In [ ]:
#### teste

#features = [query] * len (teste), [t[0] for t in teste]

QUERY = [query] * len (teste)
DOCSS = [t[0] for t in teste]
DOC_NAMES = [t[2] for t in teste]
#print (QUERY)
#print (DOCSS)
#print (DOC_NAMES)


pares = list(zip(QUERY, DOCSS))

inputs = ce_tokenization (pares, return_tensors = "pt", padding = True, truncation = True)
#print (inputs)

cross_encoder.eval()
with torch.no_grad():
    logits = cross_encoder (**inputs).logits
    #print (logits)

#Organiza os logits com os chunks correspondentes
rerank = sorted(zip(DOCSS, DOC_NAMES, logits.tolist()), key = lambda x: x[2], reverse = True)

display (rerank)

In [ ]:
feed_llm = [chunk for chunk, scores, docs in rerank [:8]]

display (feed_llm)

<h3> LLM </h3>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

path = r"C:\Users\Admin\Desktop\models\Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer2 = AutoTokenizer.from_pretrained (path)

device


In [ ]:
prompt = "Em que consiste o relatório da rede EU Kids Online ?"


chat_template = f"""
<|im_start|>
És um assistente especializado em responder exclusivamente com base no contexto fornecido.

Regras:
1. Utiliza apenas a informação do Contexto.
2. Não inventes informação.
3. Não uses conhecimento externo.
4. Se a resposta não existir no contexto, responde exatamente:
   "Informação não encontrada na base de dados."
5. Responde de forma breve e direta.
6. Formata a resposta de forma clara.

Contexto:
{feed_llm}
<|im_end|>

<|im_start>
{prompt}
<|im_end|>

<|im_start|>
"""

model_inputs = tokenizer2 (chat_template, return_tensors = "pt").to(model.device)

generated_ids = model.generate(**model_inputs, max_new_tokens = 512)

input_length = model_inputs["input_ids"].shape[1]
response_tokens = generated_ids[0][input_length:]

print(tokenizer2.decode(response_tokens, skip_special_tokens = True))

<hr>

In [ ]:
mrr_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    sparse_retr = sparse_retrieval.invoke(query)

    rr = 0

    for rank, doc in enumerate(dense_retr, start=1):
        if doc.metadata["title"] == gold_doc:
            rr = 1 / rank
            break

    mrr_scores.append(rr)

mrr = sum(mrr_scores) / len(mrr_scores)

print("MRR Sparse:", mrr)

In [ ]:
mrr_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    dense_retr = dense_retrieval.invoke(query)

    rr = 0

    for rank, doc in enumerate(dense_retr, start=1):
        if doc.metadata["title"] == gold_doc:
            rr = 1 / rank
            break

    mrr_scores.append(rr)

mrr = sum(mrr_scores) / len(mrr_scores)

print("MRR Dense:", mrr)

In [ ]:
rrf_scores = []

for item in dados:

    query = item["query"]
    gold_doc = item["doc"]

    rankings = [
        dense_retrieval.invoke(query),
        sparse_retrieval.invoke(query)
    ]

    rrf_ranked = Reciprocal_Rank_Fusion(rankings)

    rr = 0

    for rank, (doc_content, score, doc_id) in enumerate(rrf_ranked, start=1):
        if doc_id == gold_doc:
            rr = 1 / rank
            break

    rrf_scores.append(rr)

mrr_rrf = sum(rrf_scores) / len(rrf_scores)

print("MRR RRF:", mrr_rrf)